# TinyChirp CNN-Time TensorFlow

Train a 1D CNN on raw audio waveforms (no spectrogram), export an int8 TFLite model, and write Rust quantized samples in `audio_samples/<model>.rs`.

This mirrors `building_tensorflow/tinychirp_cnn.ipynb` but replaces the 2D mel CNN with a 1D time CNN similar to `StreamingCNNArch` from the TinyChirp `CNN_Time` model.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from typing import TYPE_CHECKING

from utils import (
    TARGET_AUDIO_LEN_TIME,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
)

if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

configure_tf_runtime()
set_global_seed()

MODEL_STEM = "cnn_time_tf"
paths = get_paths(MODEL_STEM)
OUT_TFLITE = paths.out_tflite
OUT_AUDIO_RS = paths.out_audio_rs
BATCH_SIZE = 32

In [ ]:
from utils import make_time_datasets

train_ds, val_ds, test_ds, label_names = make_time_datasets()
num_labels = len(label_names)
print("Classes:", label_names)

In [ ]:
model = keras.Sequential([
    # Enter as rank-2 tensor (batch, time), then expand to 4D for Conv2D.
    keras.layers.Input(shape=(TARGET_AUDIO_LEN_TIME,)),
    keras.layers.Reshape((TARGET_AUDIO_LEN_TIME, 1, 1), name="to_4d"),
    keras.layers.Conv2D(4, (3, 1), activation="relu"),
    keras.layers.AveragePooling2D(pool_size=(2, 1), strides=(2, 1)),
    keras.layers.Conv2D(8, (3, 1), activation="relu"),
    keras.layers.GlobalAveragePooling2D(keepdims=True, name="final_pool"),
    keras.layers.Flatten(name="flatten"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(num_labels),
])


model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.summary()

In [ ]:
from utils import init_wandb, get_callbacks, finish_wandb

init_wandb(MODEL_STEM, config={
    "conv1_filters": 4,
    "conv2_filters": 8,
    "kernel_size": 3,
    "dense_units": 64,
})

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    validation_steps=50,
    callbacks=get_callbacks(10,5, BATCH_SIZE)
)
finish_wandb()

In [ ]:
from utils import (
    build_representative_batches,
    export_keras_model_to_int8_tflite,
)

export_inputs = keras.Input(shape=(TARGET_AUDIO_LEN_TIME,), name="audio_waveform")
export_outputs = model(export_inputs)
export_model = keras.Model(export_inputs, export_outputs, name="cnn_time_microflow")

val_specs = build_representative_batches(test_ds, take=100)

export_keras_model_to_int8_tflite(export_model, val_specs, OUT_TFLITE)

In [ ]:
from utils import evaluate_tflite_model, SAMPLE_RATE

hyperparams = {
    "conv1_filters": 4,
    "conv1_kernel": [3, 1],
    "conv2_filters": 8,
    "conv2_kernel": [3, 1],
    "pool_size": [2, 1],
    "dense_hidden": 64,
    "target_audio_len": TARGET_AUDIO_LEN_TIME,
    "sample_rate": SAMPLE_RATE,
    "batch_size": BATCH_SIZE,
}

train_m, test_m, avg_ms = evaluate_tflite_model(
    OUT_TFLITE, MODEL_STEM, train_ds, test_ds, hyperparams=hyperparams,
)
print(f"Avg inference: {avg_ms:.3f} ms")